# Neuro-Fabric-Core — EEGConformer Training Pipeline

**Repository commit:** `5303f79`

This notebook executes the full EEGConformer training pipeline from
preprocessing through ONNX export, producing a trained checkpoint and a
parity-verified ONNX artefact ready for deployment in the Neuro-Fabric-Core
browser inference stack.

**Pipeline:**
```
BCI-IV-2a (MOABB)
   → preprocess.py   (22ch · 250Hz · 1000 samples · z-score)
   → train.py         (EEGConformer, 4-class motor imagery, AdamW + cosine)
   → validate.py      (cross-subject hold-out, subject 9)
   → evaluate.py      (intra/inter-class cosine, recall@10 vs PCA)
   → export_onnx.py   (opset 17, embedding+logits heads, parity ≥ 0.999)
   → package.py       (sha256 manifest + MODEL_CARD)
```

**Requirements:** Python 3.12, GPU runtime (T4 or better).

**Runtime:** ~30–60 minutes (dataset download + training + export).

## 1. Environment setup

In [ ]:
import sys
import subprocess

print(f"Python: {sys.version}")
major, minor = sys.version_info[:2]
if (major, minor) < (3, 10):
    raise RuntimeError(f"Python {major}.{minor} is too old. Need 3.10-3.12.")
if (major, minor) > (3, 12):
    print(f"Warning: Python {major}.{minor} - the pinned deps target 3.12. Proceeding, but watch for wheel availability errors.")
elif (major, minor) == (3, 12):
    print("OK: Python 3.12")
else:
    print(f"Note: Python {major}.{minor} - recommended is 3.12. Proceeding.")

# Check GPU
gpu_check = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                          capture_output=True, text=True)
if gpu_check.returncode == 0:
    print(f"GPU: {gpu_check.stdout.strip()}")
else:
    print("Warning: No GPU detected. Training will use CPU (much slower).")
    print("   Go to Runtime -> Change runtime type -> T4 GPU")

## 2. Clone the repository

In [ ]:
import os

REPO_URL = "https://github.com/ouadaououhoussemdroid/neuro-fabric-core.git"
COMMIT = "5303f79"
REPO_DIR = "/content/neuro-fabric-core"

if os.path.exists(REPO_DIR):
    print(f"Repository exists at {REPO_DIR}, pulling latest...")
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", COMMIT], check=True)
else:
    print(f"Cloning repository at commit {COMMIT}...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", COMMIT], check=True)

# Verify the commit
result = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"],
                        capture_output=True, text=True, check=True)
print(f"Checked out: {result.stdout.strip()[:12]}")
print(f"Repository at: {REPO_DIR}")

## 3. Install dependencies

The pinned dependency set (from `training/requirements.txt`) has been
verified end-to-end on Python 3.10–3.12 via a clean `pip install`. We install
torch first with CUDA support from the PyTorch wheel index, then the rest
of the requirements.

**Resolution rules locked in by this set** (see
`docs/audits/training-dependency-resolution-v2.md`):
- braindecode 1.1.0 uses the `BNCI2014_001` MOABB name.
- moabb 1.4.0 requires numpy ≥ 2.
- torch and torchaudio must share the same minor (2.5.1).

In [ ]:
import subprocess
import sys
import importlib.metadata as md

REQUIREMENTS_PATH = f"{REPO_DIR}/training/requirements.txt"

# Read the pinned versions from requirements.txt
pins = {}
with open(REQUIREMENTS_PATH) as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if "==" in line:
            name, ver = line.split("==", 1)
            pins[name.strip()] = ver.strip()

print(f"Pinned requirements: {len(pins)} packages")


def installed_version(pkg_name):
    """Return the installed version or None if not installed."""
    dist_name = {"sklearn": "scikit-learn"}.get(pkg_name, pkg_name)
    try:
        return md.version(dist_name)
    except md.PackageNotFoundError:
        return None


def pip_install(args, label=""):
    """Run pip install, print full stdout/stderr, return True/False."""
    cmd = [sys.executable, "-m", "pip", "install", *args]
    print("--- pip install " + " ".join(args) + " ---")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="")
    if result.returncode != 0:
        print(f"[FAILED] {label} (exit {result.returncode})")
        return False
    print(f"[OK] {label}")
    return True


# Check what is already installed and at the right version.
# Only install packages that are missing or at a different version
# -- no uninstalls, no force-upgrades of already-correct packages.
missing = []
wrong_ver = []
correct = []

for pkg, pinned_ver in pins.items():
    got = installed_version(pkg)
    got_base = got.split("+")[0] if got else None
    if got_base is None:
        missing.append(pkg)
    elif got_base == pinned_ver:
        correct.append(pkg)
    else:
        wrong_ver.append((pkg, got, pinned_ver))

print(f"Already correct: {len(correct)} packages")
if missing:
    print(f"Missing (not installed): {missing}")
if wrong_ver:
    for pkg, got, want in wrong_ver:
        print(f"  Version mismatch: {pkg}={got} -> need {want}")

# Build the install list: missing + wrong_ver packages
to_install = missing + [pkg for pkg, _, _ in wrong_ver]

# Separate torch/torchaudio (need CUDA index) from everything else.
torch_pkgs = [p for p in to_install if p in ("torch", "torchaudio")]
other_pkgs = [p for p in to_install if p not in ("torch", "torchaudio")]

# --- Install torch with CUDA support (if needed) ---
if torch_pkgs:
    print()
    torch_args = [f"{p}=={pins[p]}" for p in torch_pkgs]
    torch_args += ["--index-url", "https://download.pytorch.org/whl/cu121"]
    pip_install(torch_args, label="torch (CUDA 12.1)")
else:
    print()
    print("torch/torchaudio already at pinned version -- skipping.")

# --- Install remaining packages one by one ---
# Installing individually means a single failure does not abort the
# whole cell. The user sees the full pip output for each package.
if other_pkgs:
    print()
    print(f"=== Installing {len(other_pkgs)} packages ===")
    failed = []
    for pkg in other_pkgs:
        spec = f"{pkg}=={pins[pkg]}"
        if not pip_install([spec], label=pkg):
            failed.append(pkg)
    if failed:
        print()
        print(f"Failed packages: {failed}")
        print("The export script degrades gracefully if onnxsim/onnxoptimizer")
        print("are absent. Core packages (braindecode, moabb, mne, onnx,")
        print("onnxruntime, mlflow) must succeed for training to proceed.")
        # Check if any core packages failed
        core_pkgs = {"braindecode", "moabb", "mne", "onnx", "onnxruntime", "mlflow"}
        core_failures = [p for p in failed if p in core_pkgs]
        if core_failures:
            raise RuntimeError(f"Core packages failed: {core_failures}")
else:
    print()
    print("All dependencies already at pinned versions.")

print()
print("Dependency installation step complete.")

## 4. Compatibility check

Run `check_compat.py` to verify all pinned versions match and the
`BNCI2014_001` MOABB alias resolves (the most common failure mode).

In [ ]:
SCRIPTS_DIR = f"{REPO_DIR}/training/scripts"
os.chdir(SCRIPTS_DIR)
print(f"Working directory: {os.getcwd()}")

result = subprocess.run([sys.executable, "check_compat.py"],
                        capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Compatibility check failed. See output above.")
print("✓ Compatibility check passed.")

## 5. Download BCI-IV-2a dataset

Downloads 9 subjects of BCI Competition IV 2a via MOABB. Cached under
`training/cache/moabb/`. Safe to re-run (MOABB no-ops if files are present).

**Note:** This step can take 5–15 minutes depending on network speed.

In [ ]:
result = subprocess.run([sys.executable, "acquire_dataset.py"],
                        capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("Dataset acquisition failed.")
print("✓ Dataset downloaded.")

## 6. Preprocess

MOABB `MotorImagery` paradigm → bandpass 4–38 Hz → epoch 0–4 s →
per-trial per-channel z-score → `train.npz` + `holdout.npz`.

Contract: `[N, 22, 1000]` float32, `y` int64, `subjects` int64.
Hold-out: subject 9 (cross-subject).

In [ ]:
result = subprocess.run([sys.executable, "preprocess.py"],
                        capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("Preprocessing failed.")
print("✓ Preprocessing complete.")

## 7. Train EEGConformer

AdamW + CosineAnnealingLR + AMP + early stopping (patience 30).
Subject-stratified internal validation split (10%).

Saves the best-validation checkpoint to
`training/artefacts/eegconformer-bciiv2a-v1/eegconformer.pt`.

MLflow logging is integrated (degrades to no-op if MLflow is not
configured — the tracking URI defaults to a local SQLite file).

**Note:** With 200 epochs max and early stopping patience 30, training
typically converges in 30–80 epochs on a T4 GPU (~10–20 minutes).

In [ ]:
# Set MLflow tracking URI to a local SQLite file (optional, for experiment tracking).
# The MLflowTracker in train.py degrades to no-op if MLflow is not installed.
CACHE_DIR = f"{REPO_DIR}/training/cache"
os.makedirs(CACHE_DIR, exist_ok=True)
os.environ["MLFLOW_TRACKING_URI"] = f"sqlite:///{CACHE_DIR}/mlruns.db"

result = subprocess.run([sys.executable, "train.py"],
                        capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("Training failed.")
print("Training complete.")

## 8. Validate (cross-subject hold-out)

Loads the best checkpoint, runs inference on the subject-9 hold-out set,
and reports accuracy + per-class breakdown.

Acceptance threshold: `holdout_accuracy ≥ 0.55`.

In [ ]:
result = subprocess.run([sys.executable, "validate.py"],
                        capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("Validation failed.")
print("✓ Validation complete.")

## 9. Evaluate embedding quality

Captures 32-D embeddings from the `model.fc` head via a forward hook,
then computes:
- **Intra/inter-class cosine similarity** (mean ± std)
- **Separation margin** (intra − inter, higher is better)
- **Recall@10** vs PCA baseline
- **Embedding norms** and feature variance

Acceptance threshold: `recall_at_k["10"] ≥ 0.55` and `beats_pca == true`.

In [ ]:
result = subprocess.run([sys.executable, "evaluate.py", "--k", "10"],
                        capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("Evaluation failed.")
print("✓ Evaluation complete.")

## 10. Export to ONNX

Exports the trained EEGConformer to ONNX (opset 17) with named outputs
`embedding` (32-D) and `logits` (4-class). The export wrapper mirrors
the braindecode forward inline to survive `torch.onnx.export` with
`do_constant_folding=True` (the previous hook-based approach produced
cosine 0.30 instead of >0.999).

Post-export:
- `onnx-simplifier` + `onnxoptimizer` (optional, degrades gracefully)
- `onnx.checker.check_model` (graph validity)
- PyTorch↔ORT cosine parity check (`≥ 0.999`)

In [ ]:
result = subprocess.run([sys.executable, "export_onnx.py"],
                        capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("ONNX export failed.")
print("✓ ONNX export complete.")

## 11. Package artefacts

Bundles the trained checkpoint + ONNX + manifest (sha256, contract) +
MODEL_CARD + train_history + validation_report + evaluation_report into
`training/artefacts/eegconformer-bciiv2a-v1/`.

In [ ]:
result = subprocess.run([sys.executable, "package.py"],
                        capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("Packaging failed.")
print("✓ Packaging complete.")

## 12. Verify the trained ONNX artefact

Quick smoke test: load the exported ONNX, run a forward pass on a random
input, and verify the output shapes match the contract
(`embedding [1, 32]`, `logits [1, 4]`).

In [ ]:
import numpy as np
import onnxruntime as ort

ARTEFACTS_DIR = f"{REPO_DIR}/training/artefacts/eegconformer-bciiv2a-v1"
onnx_path = f"{ARTEFACTS_DIR}/eegconformer.onnx"

print(f"Loading: {onnx_path}")
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])

# Check I/O metadata
for inp in sess.get_inputs():
    print(f"  Input:  {inp.name} shape={inp.shape} dtype={inp.type}")
for out in sess.get_outputs():
    print(f"  Output: {out.name} shape={out.shape} dtype={out.type}")

# Forward pass on random input [1, 22, 1000]
dummy = np.random.randn(1, 22, 1000).astype(np.float32)
outputs = sess.run(None, {sess.get_inputs()[0].name: dummy})

for i, out in enumerate(sess.get_outputs()):
    print(f"  → {out.name}: shape={outputs[i].shape} mean={outputs[i].mean():.6f} std={outputs[i].std():.6f}")

assert outputs[0].shape == (1, 32), f"embedding shape mismatch: {outputs[0].shape} != (1, 32)"
assert outputs[1].shape == (1, 4), f"logits shape mismatch: {outputs[1].shape} != (1, 4)"
print("\n✓ ONNX artefact verified: embedding [1,32] + logits [1,4].")

## 13. Download artefacts

Download the trained checkpoint and ONNX artefact to your local machine.

In [ ]:
from google.colab import files
import json

# Display the manifest
manifest_path = f"{ARTEFACTS_DIR}/manifest.json"
if os.path.exists(manifest_path):
    with open(manifest_path) as f:
        manifest = json.load(f)
    print("=== Manifest ===")
    print(json.dumps(manifest, indent=2))

# Download key files
for fname in ["eegconformer.pt", "eegconformer.onnx", "manifest.json"]:
    fpath = f"{ARTEFACTS_DIR}/{fname}"
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"\nDownloading {fname} ({size_mb:.1f} MB)...")
        files.download(fpath)
    else:
        print(f"⚠ {fname} not found at {fpath}")

print("\n✓ Done. Copy eegconformer.onnx to public/models/ in the Neuro-Fabric-Core repo.")

## 14. Deploy to Neuro-Fabric-Core

After downloading `eegconformer.onnx`:

1. Copy it to `public/models/eegconformer.onnx` in your local clone.
2. Run the artefact manifest generator (build step):
   ```bash
   bun run build  # regenerates public/models/manifest.json with the new SHA-256
   ```
3. Verify the ONNX loads in the browser:
   ```bash
   bun run dev
   # Upload an EDF file → check that fellBack=false and dim=32
   ```
4. Run the evaluation workflow (see `training/docs/EVALUATION_RUNBOOK.md`)
   to confirm `recall@10 ≥ 0.55` and `beats_pca == true`.

**Acceptance criteria for GA** (from the production-readiness audit):
- `holdout_accuracy ≥ 0.55`
- `recall_at_k["10"] ≥ 0.55` (vs PCA ~0.30–0.40)
- `beats_pca == true`
- PyTorch↔ORT cosine `≥ 0.999` (checked at export time)
- `separation_margin > 0` (intra-class cosine > inter-class cosine)